# Chapter 5: Scaling Up to Real Robot Data
Leave MiniPushT behind. Load PushT (206 human teleop episodes) from the LeRobot Hub, add proprioception, and find out what the Chapter 4 architecture actually does on real data.

In [ ]:
!pip install torch torchvision numpy matplotlib transformers pillow
!pip install lerobot gym-pusht

In [ ]:
# Skips the clone if it is already present, and surfaces the real error if
# it fails, rather than hiding it and failing confusingly on the %cd below.
![ -d vla-from-scratch ] || git clone https://github.com/FanFeast/vla-from-scratch.git
%cd vla-from-scratch/chapters/05_scaling_up

## Real Data Looks Different

MiniPushT demos came from a scripted oracle: deterministic, noiseless, always optimal. PushT demos come from **humans with a mouse**. They hesitate, overshoot, take different routes to the same goal, and stop at slightly different places.

That difference is the whole chapter.

In [ ]:
from lerobot.datasets import LeRobotDataset
import matplotlib.pyplot as plt
import torch

ds = LeRobotDataset("lerobot/pusht")
print(f"Frames:   {len(ds)}")
print(f"Episodes: {ds.num_episodes}")
print(f"Keys:     {list(ds[0].keys())}")

frame = ds[0]
print(f"\nimage {tuple(frame['observation.image'].shape)}  "
      f"state {tuple(frame['observation.state'].shape)}  "
      f"action {tuple(frame['action'].shape)}")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, idx in zip(axes, [0, 40, 80, 120]):
    ax.imshow(ds[idx]["observation.image"].permute(1, 2, 0).numpy())
    ax.set_title(f"frame {idx}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Single-Pass Extraction

Decoding video frames is the expensive part -- far more than SigLIP itself. So we iterate the dataset **once**, extract vision embeddings + states + actions + episode indices together, and cache them as tensors.

Every later experiment reads the cache. This is the difference between a 20-minute chapter and a 4-hour one.

In [ ]:
from data_pipeline import extract_and_cache, PRESETS

cfg = PRESETS["pusht"]
data = extract_and_cache(
    repo_id=cfg["repo_id"],
    image_key=cfg["image_key"],
    state_key=cfg["state_key"],
    action_key=cfg["action_key"],
    cache_prefix="pusht",
    skip_embeddings=True,     # reuse cache if present
)

embeddings = data["embeddings"]
states = data["states"]
actions = data["actions"]
episode_indices = data["episode_indices"]

print(f"embeddings {tuple(embeddings.shape)}")
print(f"states     {tuple(states.shape)}")
print(f"actions    {tuple(actions.shape)}")

## Split by Episode, Not by Frame

A random frame split would leak: frame 501 and frame 502 are nearly the same image, so half of any "validation" frame's neighbours sit in training. Validation loss would look great and mean nothing.

Splitting whole **episodes** keeps the sets genuinely disjoint. Normalization stats also come from the training split only.

In [ ]:
from data_pipeline import split_by_episode, compute_norm_stats, RobotDataset

train_mask, val_mask = split_by_episode(episode_indices, val_ratio=0.13)
print(f"train frames {int(train_mask.sum())}   val frames {int(val_mask.sum())}")
print(f"train episodes {len(episode_indices[train_mask].unique())}   "
      f"val episodes {len(episode_indices[val_mask].unique())}")

action_stats = compute_norm_stats(actions[train_mask])
state_stats = compute_norm_stats(states[train_mask])
print(f"\nRaw action range: {action_stats.min_val.tolist()} .. {action_stats.max_val.tolist()}")
print("(PushT actions are pixel coordinates in [0, 512] -- normalized to [-1, 1])")

norm_actions = action_stats.normalize(actions)
norm_states = state_stats.normalize(states)

## Train the Same Six Configurations

Identical heads to Chapter 4, plus a `ProprioEncoder` that projects robot state to 64 dims and concatenates it with SigLIP's 768 -- 832 dims in.

We use fewer epochs here than the published run so the notebook finishes in one sitting. The README table is the full 200-epoch result; the conclusion is the same either way.

In [ ]:
from pathlib import Path
from data_pipeline import build_vla, train, evaluate_offline, HEAD_TYPES, CHUNK_SIZES, DEVICE

EPOCHS = 30          # published run uses 200 -- see README
ckpt_dir = Path("checkpoints"); ckpt_dir.mkdir(exist_ok=True)
print(f"device: {DEVICE}")

results, histories = {}, {}

for head in HEAD_TYPES:
    for cs in CHUNK_SIZES:
        name = f"{head}_K{cs}"
        print(f"\n--- {name} ---")

        train_ds = RobotDataset(embeddings[train_mask], norm_states[train_mask],
                                norm_actions[train_mask], episode_indices[train_mask], cs)
        val_ds = RobotDataset(embeddings[val_mask], norm_states[val_mask],
                              norm_actions[val_mask], episode_indices[val_mask], cs)

        model = build_vla(head, cfg["state_dim"], cfg["action_dim"], cs)
        histories[name] = train(model, train_ds, val_ds, epochs=EPOCHS,
                                batch_size=cfg["batch_size"], lr=cfg["lr"],
                                patience=cfg["patience"],
                                ckpt_path=ckpt_dir / f"pusht_{name}.pt", device=DEVICE)

        results[name] = evaluate_offline(model, val_ds, DEVICE)
        print(f"  val_loss={results[name]['val_loss']:.4f}  MAE={results[name]['mae']:.4f}")

## Offline Metrics Look Fine

In [ ]:
print(f"{'Config':<16} {'Val Loss':>10} {'MAE':>8}")
print("-" * 36)
for name, r in results.items():
    print(f"{name:<16} {r['val_loss']:>10.4f} {r['mae']:>8.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
for name, h in histories.items():
    if h.get("val_loss"):
        ax.plot(h["val_loss"], label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Val loss"); ax.set_yscale("log")
ax.set_title("PushT validation loss (log scale)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Now Actually Run the Policy

This is the cell that matters. Offline MAE says regression is nailing it. Closed-loop rollout in `gym-pusht` says otherwise.

Each step re-encodes the live frame with SigLIP, so this is slower than the training loop.

In [ ]:
from data_pipeline import evaluate_live_pusht

best = min(results, key=lambda n: results[n]["mae"])
print(f"Rolling out the best offline model: {best} (MAE={results[best]['mae']:.4f})\n")

model = build_vla(best.split("_K")[0], cfg["state_dim"], cfg["action_dim"], int(best.split("_K")[1]))
model.load_state_dict(torch.load(ckpt_dir / f"pusht_{best}.pt", map_location="cpu", weights_only=True))

live = evaluate_live_pusht(model, state_stats, action_stats, n_episodes=20, device=DEVICE)
print(f"Live success rate: {live['success_rate']*100:.0f}%")
print(f"Mean episode length: {live['avg_length']:.0f} steps (cap is 300)")

## What We Learned

**Low loss is not task success.** Regression reaches MAE ~0.05 on held-out frames and still scores **0%** closed-loop. Both numbers are correct; they measure different things. Per-frame MAE asks "is this action close to what the human did *here*?" Rollout asks "does a chain of 300 of your own actions reach the goal?" Small errors compound -- the policy drifts into states no human demo ever visited, where its predictions were never trained to be right.

**Human data is multi-modal, and MSE averages it.** When two demonstrators push the T-block around opposite sides, the mean of those trajectories goes straight through the block. Regression learns that mean and executes something no human ever did. This is *the* argument for generative action heads, and it is why Chapter 6 exists.

**Discrete degrades at high dimension.** 14-D ALOHA becomes 14 x 256 independent classification problems per timestep, with no coupling between joints.

**Diffusion has the best val loss on ALOHA (0.070) and still 0% success** -- a 2-layer MLP denoiser with 10 inference steps cannot represent the score function. The idea is right; the capacity is not.

Nothing here is a bug. This is the honest baseline that motivates the rest of the book.

**Next:** Chapter 6 replaces the MLP denoiser with a ~12M-param transformer action expert trained by flow matching -- the first architecture in this repo that actually works on real data.